# Preprocessing

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

telco = pd.read_csv('C:\\Users\\Abdelrahman\\Desktop\\churn\\data\\WA_Fn-UseC_-Telco-Customer-Churn.csv')
telco.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


Data Cleaning

In [4]:
telco['TotalCharges'] = pd.to_numeric(telco['TotalCharges'], errors='coerce')

telco['TotalCharges'] = telco['TotalCharges'].fillna(0)

print(f"Missing values in TotalCharges: {telco['TotalCharges'].isnull().sum()}")

Missing values in TotalCharges: 0


Feature Engineering

In [5]:
telco['Churn'] = telco['Churn'].map({'Yes': 1, 'No': 0})

service_cols = ['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 
                'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

telco['TotalServices'] = (telco[service_cols] == 'Yes').sum(axis=1)

telco['AvgMonthlySpend'] = telco['TotalCharges'] / (telco['tenure'] + 1)

# Encoding 

In [6]:
if 'customerID' in telco.columns:
    telco.drop('customerID', axis=1, inplace=True)
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
for col in binary_cols:
    telco[col] = telco[col].map({'Yes': 1, 'No': 0})
categorical_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 
                    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 
                    'Contract', 'PaymentMethod']

telco_final = pd.get_dummies(telco, columns=categorical_cols, drop_first=True)

telco_final.head()


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,Female,0,1,0,1,0,1,29.85,29.85,0,...,False,False,False,False,False,False,False,False,True,False
1,Male,0,0,0,34,1,0,56.95,1889.50,0,...,False,False,False,False,False,True,False,False,False,True
2,Male,0,0,0,2,1,1,53.85,108.15,1,...,False,False,False,False,False,False,False,False,False,True
3,Male,0,0,0,45,0,0,42.30,1840.75,0,...,True,False,False,False,False,True,False,False,False,False
4,Female,0,0,0,2,1,1,70.70,151.65,1,...,False,False,False,False,False,False,False,False,True,False


# Scaleing

In [7]:
scaler = StandardScaler()
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'TotalServices', 'AvgMonthlySpend']

telco_final[num_cols] = scaler.fit_transform(telco_final[num_cols])

print("Preprocessing Complete!")
print(f"Final Data Shape: {telco_final.shape}")

Preprocessing Complete!
Final Data Shape: (7043, 33)


In [8]:
print(telco.dtypes.value_counts())
telco.to_csv('Telco_Churn_clean.csv', index=False)

str        11
int64       8
float64     3
Name: count, dtype: int64
